# Paired-book pressure redundancy diagnostic

A label-free reconstruction of the abandoned July 15 PolyMomentum paired-book capture.

## tl;dr

Across 6,465 valid one-second paired-book states from 24 markets, the proposed `chosen pressure > 0 and opposite pressure < 0` predicate disagreed with `chosen pressure > 0` zero times in 12,930 orientation comparisons. The observed cross-touch mirror error was also exactly zero. This rejects only the opposite-book clause as an independent feature in this diagnostic; it does **not** establish whether chosen-token pressure predicts terminal settlement or improves profitability. The active forward rule remains unchanged.

## Context & Methods

The active `binary_complement_coherence_v1` rule remains frozen and its forward outcomes remain sealed. This diagnostic asks whether a proposed backup—requiring the chosen book's microprice pressure to point up while the opposite book points down—adds information beyond chosen-token pressure. The analysis uses only contemporaneous order-book state; terminal labels are never loaded into the calculation.

### Key assumptions

- Sample the causal state at the end of each one-second interval in each captured five-minute market.
- Reconstruct the native-order incremental book using the event's authoritative best bid/ask, then apply the same top-three aggregation and pressure formula as the Rust microstructure implementation.
- Require a full snapshot for both tokens, valid interior books, positive top-three depth, and at most 30 seconds of age for each token.
- Treat this as retrospective structural evidence only. The BTC and Chainlink tapes contain internal gaps, so this capture is forbidden as promotion or exact-replay evidence.
- Native source timestamps regress 865 times within condition. The reconstruction follows native event order and exposes this limitation; it is not a sub-second execution-path parity test.

In [1]:
from __future__ import annotations

import gzip
import hashlib
import json
import math
import statistics
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/Users/ttoomm/Documents/PolyMomentum')
CAPTURE = Path('/private/tmp/fresh-block-canary-recovered/segment_001')
CONVERTED = CAPTURE / 'converted_v10'
RAW = CAPTURE / 'raw'
EVENT_FILES = [CONVERTED / f'2026-07-15T{hour:02d}.v1.candles.jsonl.gz' for hour in (6, 7, 8)]
MANIFEST_PATH = CONVERTED / 'manifest.json'
RESOLUTION_PATH = CONVERTED / 'resolution_manifest.json'
RUST_MICROSTRUCTURE_PATH = ROOT / 'rust_engine/src/strategy/microstructure.rs'
OUTPUT_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/20260721_paired_book_pressure_redundancy_diagnostic.json'
DEPTH_LEVELS = 3
MAX_BOOK_AGE_SECONDS = 30.0
EPSILON = 1e-12

manifest = json.loads(MANIFEST_PATH.read_text())
resolution = json.loads(RESOLUTION_PATH.read_text())
assert manifest['stats']['skipped_malformed_raw'] == 0
assert manifest['stats']['skipped_unknown_market'] == 0
assert manifest['stats']['skipped_unknown_token'] == 0
assert resolution['stats']['terminal'] == 24

markets = {row['condition_id']: row for row in resolution['markets']}
tokens_by_condition = defaultdict(dict)
for token_id, token_meta in manifest['token_outcomes'].items():
    tokens_by_condition[token_meta['condition_id']][token_meta['outcome'].lower()] = token_id
assert set(tokens_by_condition) == set(markets)
assert all(set(tokens) == {'up', 'down'} for tokens in tokens_by_condition.values())

min_open = int(min(row['open_ts_s'] for row in markets.values()))
max_close = int(max(row['close_ts_s'] for row in markets.values()))
condition_by_second = {}
for condition_id, row in markets.items():
    for second in range(int(row['open_ts_s']), int(row['close_ts_s'])):
        assert second not in condition_by_second
        condition_by_second[second] = condition_id
assert len(condition_by_second) == 24 * 300

print({
    'conditions': len(markets),
    'possible_one_hz_samples': len(condition_by_second),
    'event_files': [path.name for path in EVENT_FILES],
    'source_settlement_gate': resolution['a_plus_gate']['verdict'],
})

{'conditions': 24, 'possible_one_hz_samples': 7200, 'event_files': ['2026-07-15T06.v1.candles.jsonl.gz', '2026-07-15T07.v1.candles.jsonl.gz', '2026-07-15T08.v1.candles.jsonl.gz'], 'source_settlement_gate': 'BTC_TAPE_INTERNAL_GAP'}


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def quantile(values: list[float], probability: float) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * probability
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1.0 - weight) + ordered[upper] * weight


def pearson(left: list[float], right: list[float]) -> float | None:
    if len(left) != len(right) or len(left) < 2:
        return None
    left_mean = statistics.fmean(left)
    right_mean = statistics.fmean(right)
    numerator = sum((a - left_mean) * (b - right_mean) for a, b in zip(left, right))
    left_ss = sum((a - left_mean) ** 2 for a in left)
    right_ss = sum((b - right_mean) ** 2 for b in right)
    denominator = math.sqrt(left_ss * right_ss)
    return numerator / denominator if denominator > 0 else None


def csv_internal_gaps(path: Path, threshold_ms: int = 5_000) -> list[dict]:
    gaps = []
    with path.open() as handle:
        header = next(handle).strip().split(',')
        timestamp_index = header.index('timestamp_ms')
        previous = None
        for line in handle:
            parts = line.rstrip().split(',')
            timestamp = int(parts[timestamp_index])
            if previous is not None and timestamp - previous > threshold_ms:
                gaps.append({'start_ms': previous, 'end_ms': timestamp, 'gap_ms': timestamp - previous})
            previous = timestamp
    return gaps


def fresh_book_state() -> dict:
    return {
        'bids': {},
        'asks': {},
        'best_bid': 0.0,
        'best_ask': 0.0,
        'last_update': None,
        'has_snapshot': False,
    }


def apply_event(books: dict[str, dict], event: dict) -> None:
    if event['ev'] == 'trade':
        return
    token = event['tok']
    book = books[token]
    timestamp = float(event['ts'])
    if event['ev'] == 'book':
        book['bids'] = {float(price): float(size) for price, size in event['bids'] if float(size) > 0}
        book['asks'] = {float(price): float(size) for price, size in event['asks'] if float(size) > 0}
        book['has_snapshot'] = True
    elif event['ev'] == 'chg':
        side = 'bids' if event['s'] == 'BUY' else 'asks'
        price = float(event['p'])
        size = float(event['sz'])
        if size > 0:
            book[side][price] = size
        else:
            book[side].pop(price, None)
    else:
        raise AssertionError(f"unknown event type: {event['ev']}")
    book['best_bid'] = float(event['bb'])
    book['best_ask'] = float(event['ba'])
    book['last_update'] = timestamp


def book_metrics(book: dict, sample_cutoff: float) -> dict | None:
    if not book['has_snapshot'] or book['last_update'] is None:
        return None
    age = sample_cutoff - book['last_update']
    if age < -1e-9 or age > MAX_BOOK_AGE_SECONDS:
        return None
    best_bid = book['best_bid']
    best_ask = book['best_ask']
    if not (0.0 < best_bid < best_ask < 1.0):
        return None
    bid_levels = sorted(
        ((price, size) for price, size in book['bids'].items() if size > 0 and price <= best_bid + 1e-9),
        reverse=True,
    )[:DEPTH_LEVELS]
    ask_levels = sorted(
        ((price, size) for price, size in book['asks'].items() if size > 0 and price >= best_ask - 1e-9)
    )[:DEPTH_LEVELS]
    bid_depth = sum(size for _, size in bid_levels)
    ask_depth = sum(size for _, size in ask_levels)
    if bid_depth <= 0 or ask_depth <= 0:
        return None
    spread = best_ask - best_bid
    midpoint = (best_bid + best_ask) / 2.0
    microprice = (best_ask * bid_depth + best_bid * ask_depth) / (bid_depth + ask_depth)
    pressure = max(-1.0, min(1.0, (microprice - midpoint) / (spread / 2.0)))
    return {
        'best_bid': best_bid,
        'best_ask': best_ask,
        'bid_depth': bid_depth,
        'ask_depth': ask_depth,
        'midpoint': midpoint,
        'microprice': microprice,
        'pressure': pressure,
        'age_s': age,
    }

In [3]:
books = defaultdict(fresh_book_state)
samples = []
event_counts = Counter()
out_of_order_by_condition = Counter()
last_timestamp_by_condition = defaultdict(lambda: float('-inf'))
next_second_by_condition = {condition_id: int(row['open_ts_s']) for condition_id, row in markets.items()}

def capture_second(condition_id: str, second: int) -> None:
    assert condition_by_second.get(second) == condition_id
    cutoff = second + 1.0 - 1e-9
    token_pair = tokens_by_condition[condition_id]
    up = book_metrics(books[token_pair['up']], cutoff)
    down = book_metrics(books[token_pair['down']], cutoff)
    if up is None or down is None:
        samples.append({'condition_id': condition_id, 'second': second, 'valid': False})
        return
    samples.append({
        'condition_id': condition_id,
        'second': second,
        'valid': True,
        'up_pressure': up['pressure'],
        'down_pressure': down['pressure'],
        'up_microprice': up['microprice'],
        'down_microprice': down['microprice'],
        'up_midpoint': up['midpoint'],
        'down_midpoint': down['midpoint'],
        'up_best_bid': up['best_bid'],
        'up_best_ask': up['best_ask'],
        'down_best_bid': down['best_bid'],
        'down_best_ask': down['best_ask'],
        'up_bid_depth': up['bid_depth'],
        'up_ask_depth': up['ask_depth'],
        'down_bid_depth': down['bid_depth'],
        'down_ask_depth': down['ask_depth'],
    })

for event_path in EVENT_FILES:
    with gzip.open(event_path, 'rt') as handle:
        for line in handle:
            event = json.loads(line)
            condition_id = event['mkt']
            if condition_id not in markets:
                continue
            timestamp = float(event['ts'])
            if timestamp + 1e-9 < last_timestamp_by_condition[condition_id]:
                out_of_order_by_condition[condition_id] += 1
            last_timestamp_by_condition[condition_id] = max(last_timestamp_by_condition[condition_id], timestamp)
            close_second = int(markets[condition_id]['close_ts_s'])
            while (
                next_second_by_condition[condition_id] < close_second
                and next_second_by_condition[condition_id] + 1.0 - 1e-9 < timestamp
            ):
                second = next_second_by_condition[condition_id]
                capture_second(condition_id, second)
                next_second_by_condition[condition_id] += 1
            apply_event(books, event)
            event_counts[event['ev']] += 1

for condition_id, row in markets.items():
    close_second = int(row['close_ts_s'])
    while next_second_by_condition[condition_id] < close_second:
        second = next_second_by_condition[condition_id]
        capture_second(condition_id, second)
        next_second_by_condition[condition_id] += 1

assert len(samples) == len(condition_by_second)
valid_samples = [row for row in samples if row['valid']]
print({
    'events': dict(event_counts),
    'within_condition_native_order_regressions': sum(out_of_order_by_condition.values()),
    'samples': len(samples),
    'valid_samples': len(valid_samples),
    'valid_coverage': len(valid_samples) / len(samples),
})

{'events': {'book': 120475, 'chg': 7561554}, 'within_condition_native_order_regressions': 865, 'samples': 7200, 'valid_samples': 6465, 'valid_coverage': 0.8979166666666667}


In [4]:
up_pressures = [row['up_pressure'] for row in valid_samples]
down_pressures = [row['down_pressure'] for row in valid_samples]
pressure_sum_abs = [abs(row['up_pressure'] + row['down_pressure']) for row in valid_samples]
microprice_sum_abs = [abs(row['up_microprice'] + row['down_microprice'] - 1.0) for row in valid_samples]
midpoint_sum_abs = [abs(row['up_midpoint'] + row['down_midpoint'] - 1.0) for row in valid_samples]
cross_touch_abs = [
    max(
        abs(row['up_best_bid'] + row['down_best_ask'] - 1.0),
        abs(row['up_best_ask'] + row['down_best_bid'] - 1.0),
    )
    for row in valid_samples
]
depth_mirror_abs = [
    max(
        abs(row['up_bid_depth'] - row['down_ask_depth']),
        abs(row['up_ask_depth'] - row['down_bid_depth']),
    )
    for row in valid_samples
]
non_neutral = [row for row in valid_samples if abs(row['up_pressure']) > EPSILON and abs(row['down_pressure']) > EPSILON]
opposite_sign = [row for row in non_neutral if row['up_pressure'] * row['down_pressure'] < 0]
same_sign = [row for row in non_neutral if row['up_pressure'] * row['down_pressure'] > 0]
candidate_disagreements = 0
candidate_comparisons = 0
chosen_positive_count = 0
paired_candidate_count = 0
for row in valid_samples:
    for chosen, opposite in (('up_pressure', 'down_pressure'), ('down_pressure', 'up_pressure')):
        candidate = row[chosen] > EPSILON and row[opposite] < -EPSILON
        chosen_only = row[chosen] > EPSILON
        chosen_positive_count += int(chosen_only)
        paired_candidate_count += int(candidate)
        candidate_disagreements += int(candidate != chosen_only)
        candidate_comparisons += 1

valid_conditions = sorted({row['condition_id'] for row in valid_samples})
per_condition = []
for condition_id in sorted(markets, key=lambda cid: markets[cid]['open_ts_s']):
    condition_rows = [row for row in valid_samples if row['condition_id'] == condition_id]
    per_condition.append({
        'condition_id': condition_id,
        'open_ts_s': int(markets[condition_id]['open_ts_s']),
        'valid_samples': len(condition_rows),
        'coverage': len(condition_rows) / 300.0,
        'max_abs_pressure_sum': max((abs(row['up_pressure'] + row['down_pressure']) for row in condition_rows), default=None),
        'max_abs_microprice_sum_residual': max((abs(row['up_microprice'] + row['down_microprice'] - 1.0) for row in condition_rows), default=None),
    })

source_hashes = {path.name: sha256_file(path) for path in [*EVENT_FILES, MANIFEST_PATH, RESOLUTION_PATH]}
source_hashes[str(RUST_MICROSTRUCTURE_PATH.relative_to(ROOT))] = sha256_file(RUST_MICROSTRUCTURE_PATH)
binance_gaps = csv_internal_gaps(RAW / 'binance_btcusdt_rtds.csv')
chainlink_gaps = csv_internal_gaps(RAW / 'chainlink_btcusd.csv')

structural_results = {
    'possible_one_hz_samples': len(samples),
    'valid_paired_samples': len(valid_samples),
    'valid_paired_sample_coverage': len(valid_samples) / len(samples),
    'conditions_with_valid_pair': len(valid_conditions),
    'pressure_pearson_correlation': pearson(up_pressures, down_pressures),
    'non_neutral_pressure_samples': len(non_neutral),
    'opposite_sign_pressure_samples': len(opposite_sign),
    'same_sign_pressure_samples': len(same_sign),
    'opposite_sign_rate_among_non_neutral': len(opposite_sign) / len(non_neutral) if non_neutral else None,
    'candidate_vs_chosen_positive_disagreement_count': candidate_disagreements,
    'candidate_vs_chosen_positive_comparisons': candidate_comparisons,
    'candidate_vs_chosen_positive_disagreement_rate': candidate_disagreements / candidate_comparisons,
    'chosen_positive_count': chosen_positive_count,
    'paired_candidate_count': paired_candidate_count,
    'abs_pressure_sum': {
        'median': quantile(pressure_sum_abs, 0.5),
        'p95': quantile(pressure_sum_abs, 0.95),
        'p99': quantile(pressure_sum_abs, 0.99),
        'max': max(pressure_sum_abs),
    },
    'abs_midpoint_sum_residual': {
        'median': quantile(midpoint_sum_abs, 0.5),
        'p99': quantile(midpoint_sum_abs, 0.99),
        'max': max(midpoint_sum_abs),
    },
    'abs_microprice_sum_residual': {
        'median': quantile(microprice_sum_abs, 0.5),
        'p99': quantile(microprice_sum_abs, 0.99),
        'max': max(microprice_sum_abs),
    },
    'abs_cross_touch_mirror_error': {
        'median': quantile(cross_touch_abs, 0.5),
        'p99': quantile(cross_touch_abs, 0.99),
        'max': max(cross_touch_abs),
    },
    'abs_top3_depth_mirror_error': {
        'median': quantile(depth_mirror_abs, 0.5),
        'p99': quantile(depth_mirror_abs, 0.99),
        'max': max(depth_mirror_abs),
    },
}

structural_identity_checks = {
    'all_conditions_represented': len(valid_conditions) == len(markets) == 24,
    'valid_state_population_is_nonempty': len(valid_samples) > 0,
    'cross_touch_mirror_error_at_float_tolerance': max(cross_touch_abs) <= EPSILON,
    'paired_predicate_has_zero_observed_disagreements': candidate_disagreements == 0,
    'paired_and_chosen_positive_counts_match': paired_candidate_count == chosen_positive_count,
}
structural_identity_checks['all_pass'] = all(structural_identity_checks.values())
assert structural_identity_checks['all_pass'], structural_identity_checks

evidence = {
    'schema_version': 2,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'status': 'DIAGNOSTIC_ONLY_REJECT_OPPOSITE_PRESSURE_CLAUSE_ACTIVE_RULE_UNCHANGED',
    'decision_question': 'Does opposite-book pressure add independent causal information beyond chosen-token pressure?',
    'source_authority': {
        'capture': 'abandoned fresh-block-canary captured 2026-07-15T06:49Z through 2026-07-15T08:50Z',
        'capture_conditions': 24,
        'distilled_files': [path.name for path in EVENT_FILES],
        'sha256': source_hashes,
        'terminal_labels_used_in_primary_analysis': False,
    },
    'data_quality': {
        'raw_frames': manifest['stats']['raw_frames'],
        'book_events': manifest['stats']['book_events'],
        'change_events': manifest['stats']['change_events'],
        'skipped_malformed_raw': manifest['stats']['skipped_malformed_raw'],
        'skipped_unknown_market': manifest['stats']['skipped_unknown_market'],
        'skipped_unknown_token': manifest['stats']['skipped_unknown_token'],
        'within_condition_native_order_regressions_observed': sum(out_of_order_by_condition.values()),
        'binance_internal_gaps_over_5s': binance_gaps,
        'chainlink_internal_gaps_over_5s': chainlink_gaps,
        'promotion_or_exact_replay_eligible': False,
        'quality_assessment': 'SHARE_WITH_CAVEATS_FOR_MECHANISM_SCREEN_ONLY',
        'valid_paired_sample_coverage': len(valid_samples) / len(samples),
        'invalid_paired_samples': len(samples) - len(valid_samples),
        'reason': 'reference-tape internal gaps, insufficient first-half signal preroll, timestamp regressions, and incomplete paired-state coverage; structural book-state diagnosis only',
    },
    'methodology': {
        'sampling': 'end-of-second causal state for each of 24 non-overlapping five-minute conditions',
        'book_depth_levels': DEPTH_LEVELS,
        'maximum_book_age_seconds': MAX_BOOK_AGE_SECONDS,
        'microprice': '(best_ask * bid_depth + best_bid * ask_depth) / (bid_depth + ask_depth)',
        'pressure': 'clamp((microprice - midpoint) / (spread / 2), -1, 1)',
        'pressure_identity': 'for a valid spread, pressure algebraically equals (bid_depth - ask_depth) / (bid_depth + ask_depth)',
        'rust_formula_parity': 'same top-three depth, microprice, midpoint, and pressure formulas as rust_engine/src/strategy/microstructure.rs; not an execution-timing parity claim',
        'primary_test': 'compare Up-token pressure with Down-token pressure without loading terminal labels',
    },
    'structural_results': structural_results,
    'structural_identity_checks': structural_identity_checks,
    'per_condition_quality': per_condition,
    'mechanism_assessment': {
        'opposite_book_pressure_clause': 'REJECT_AS_NON_INDEPENDENT_IN_OBSERVED_VALID_STATES',
        'chosen_token_pressure_family': 'TERMINAL_PREDICTION_AND_ECONOMIC_VALUE_NOT_EVALUATED_BY_THIS_DIAGNOSTIC',
        'reason': 'Polymarket complementary orders produced mirrored cross-touches and opposing pressure: requiring chosen pressure > 0 and opposite pressure < 0 added zero observed decisions beyond chosen pressure > 0 across the valid sampled states.',
        'scope_boundary': 'This is an observed structural equivalence, not a universal venue invariant, terminal-outcome test, strategy backtest, or profitability result.',
        'independent_information_demonstrated': False,
        'new_preregistration_created': False,
        'active_binary_complement_rule_changed': False,
        'multiplicity_action': 'do not add or score the opposite-book clause as a separate feature family',
    },
    'primary_sources': [
        {
            'title': 'Polymarket Prices & Orderbook',
            'url': 'https://docs.polymarket.com/concepts/prices-orderbook',
            'use': 'Complementary Yes and No buy orders whose prices sum to one are matched by minting a complete set.',
        },
        {
            'title': 'Polymarket CTF Exchange V2',
            'url': 'https://github.com/Polymarket/ctf-exchange-v2',
            'use': 'The official exchange implements MINT for two complementary buy orders and MERGE for two complementary sell orders.',
        },
        {
            'title': 'Queue Imbalance as a One-Tick-Ahead Price Predictor in a Limit Order Book',
            'url': 'https://arxiv.org/abs/1512.03492',
            'use': 'Queue imbalance is supported as a short-horizon next-price descriptor, not as a terminal settlement predictor.',
        },
        {
            'title': 'The Micro-Price: A High Frequency Estimator of Future Prices',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2970694',
            'use': 'Microprice is an imbalance-adjusted short-horizon price estimator.',
        },
    ],
    'decision': {
        'current_strategy_grade': 'A-',
        'a_plus_claim': False,
        'profitability_claim': False,
        'live_ready': False,
        'live_trading': 'OFF',
        'next_evidence': 'continue the unchanged binary_complement_coherence_v1 sealed block to the fixed 750-condition floor and score once',
    },
}

temporary = OUTPUT_PATH.with_name(f'{OUTPUT_PATH.name}.tmp.{__import__("os").getpid()}')
temporary.write_text(json.dumps(evidence, indent=2, sort_keys=True) + '\n')
temporary.replace(OUTPUT_PATH)

print(json.dumps({
    'valid_paired_samples': structural_results['valid_paired_samples'],
    'coverage': structural_results['valid_paired_sample_coverage'],
    'pressure_correlation': structural_results['pressure_pearson_correlation'],
    'opposite_sign_rate': structural_results['opposite_sign_rate_among_non_neutral'],
    'candidate_disagreement_rate': structural_results['candidate_vs_chosen_positive_disagreement_rate'],
    'max_abs_pressure_sum': structural_results['abs_pressure_sum']['max'],
    'output': str(OUTPUT_PATH.relative_to(ROOT)),
}, indent=2))

{
  "valid_paired_samples": 6465,
  "coverage": 0.8979166666666667,
  "pressure_correlation": -0.9999990426497684,
  "opposite_sign_rate": 0.99984532095901,
  "candidate_disagreement_rate": 0.0,
  "max_abs_pressure_sum": 0.04501844995696832,
  "output": "deploy/promotions/evidence/strategy_registry/20260721_paired_book_pressure_redundancy_diagnostic.json"
}


## Results

The 24 markets supply 7,200 possible one-second states. Both books were valid under the snapshot, interior-price, positive-depth, and 30-second-age rules in 6,465 states (89.79%); all 24 markets contributed valid states. Across two possible chosen-token orientations per state, the paired predicate and chosen-positive predicate each fired 6,464 times and disagreed 0 / 12,930 times. Up/down pressure correlation was -0.9999990, and 6,464 / 6,465 non-neutral states had opposite signs. Cross-touch mirror error was zero at median, p99, and maximum.

The evidence is not frictionless: 735 possible states were invalid, native event timestamps regressed 865 times within condition, one non-neutral state had both pressures negative, and the maximum pressure-sum residual was 0.045 even though its p99 was approximately 4.44e-14. These exceptions are why the conclusion is limited to the observed predicate decisions and is not promoted to a universal identity or execution guarantee.

In [5]:
# Bounded condition-level quality table. No outcomes are displayed or used.
for row in per_condition:
    print(
        datetime.fromtimestamp(row['open_ts_s'], tz=timezone.utc).strftime('%Y-%m-%d %H:%MZ'),
        row['valid_samples'],
        f"{row['coverage']:.1%}",
        f"pressure_sum_max={row['max_abs_pressure_sum']:.3g}" if row['max_abs_pressure_sum'] is not None else 'pressure_sum_max=n/a',
    )

2026-07-15 06:50Z 266 88.7% pressure_sum_max=5e-14
2026-07-15 06:55Z 240 80.0% pressure_sum_max=0.00388
2026-07-15 07:00Z 297 99.0% pressure_sum_max=4.44e-14
2026-07-15 07:05Z 297 99.0% pressure_sum_max=3.89e-14
2026-07-15 07:10Z 283 94.3% pressure_sum_max=4.44e-14
2026-07-15 07:15Z 272 90.7% pressure_sum_max=5.27e-14
2026-07-15 07:20Z 149 49.7% pressure_sum_max=2.41e-13
2026-07-15 07:25Z 200 66.7% pressure_sum_max=4.44e-14
2026-07-15 07:30Z 294 98.0% pressure_sum_max=4.72e-14
2026-07-15 07:35Z 287 95.7% pressure_sum_max=5e-14
2026-07-15 07:40Z 299 99.7% pressure_sum_max=0.00471
2026-07-15 07:45Z 299 99.7% pressure_sum_max=4.94e-14
2026-07-15 07:50Z 297 99.0% pressure_sum_max=4.44e-14
2026-07-15 07:55Z 281 93.7% pressure_sum_max=0.045
2026-07-15 08:00Z 217 72.3% pressure_sum_max=4.67e-14
2026-07-15 08:05Z 204 68.0% pressure_sum_max=4.71e-14
2026-07-15 08:10Z 294 98.0% pressure_sum_max=0.00346
2026-07-15 08:15Z 284 94.7% pressure_sum_max=4.91e-14
2026-07-15 08:20Z 284 94.7% pressure_sum

## Takeaways

1. Do not create a separate strategy family whose only novelty is the opposite-pressure clause; it added no observed decision information beyond chosen-positive pressure.
2. Do not infer that chosen-token pressure is good or bad for settlement prediction: this label-free diagnostic cannot answer that question.
3. Do not change `binary_complement_coherence_v1` while its sealed 750-condition block is collecting. Continue that fixed test and score it exactly once at the registered floor.
4. Keep live trading off and preserve the A- grade until the registered classification, fee-aware unit-economics, chronology, breadth, burst, and tail gates pass on fresh evidence.